<a href="https://colab.research.google.com/github/GilbertOwen/reefscan-id/blob/noaa-check/notebooks/03_eda_noaa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ReefScan-ID: NOAA bleaching dataset check

Check of the NOAA PIFSC ESD Coral Bleaching dataset before it is used for stage 2. Run in Google Colab; no GPU is needed.

### Setup: libraries

In [3]:
import huggingface_hub
import pandas as pd

print("huggingface_hub version: {}".format(huggingface_hub.__version__))
print("pandas version: {}".format(pd.__version__))

huggingface_hub version: 1.33.0
pandas version: 2.2.3


### Setup: Google Drive

In [4]:
import os
from google.colab import drive

drive.mount("/content/drive")

base_dir = "/content/drive/MyDrive/ReefScan-ID"
noaa_dir = base_dir + "/RawData/NOAA-PIFSC-ESD-CORAL-Bleaching-Dataset"
print("NOAA folder found: {}".format(os.path.isdir(noaa_dir)))
print("Contents: {}".format(sorted(os.listdir(noaa_dir))))

Mounted at /content/drive
NOAA folder found: True
Contents: ['.git', '.gitattributes', '.gitignore', 'README.md', 'test', 'train', 'val']


## Section 1: Do the labels load?

Question: does every image in the Drive copy of the dataset get one of the two labels (CORAL and CORAL_BL) from its folder name, with the counts the folders on the Hub hold?

Checks: count the image files in each split and class folder of the repository on the Hub, list the images in the Drive folder with the split and label taken from the path, then compare the counts per folder and print the label counts per split.

### Files in the repository

In [5]:
from collections import Counter
from huggingface_hub import list_repo_files

repo_id = "NMFS-OSI/NOAA-PIFSC-ESD-CORAL-Bleaching-Dataset"
image_endings = (".jpg", ".jpeg", ".png")

files = list_repo_files(repo_id, repo_type="dataset")
image_files = [name for name in files if name.lower().endswith(image_endings)]
other_files = [name for name in files if not name.lower().endswith(image_endings)]

print("Files in the repository: {}".format(len(files)))
print("Image files: {}".format(len(image_files)))
print("Other files: {}".format(other_files[:20]))

folder_counts = Counter()
for name in image_files:
    folder_counts["/".join(name.split("/")[:-1])] += 1

for folder in sorted(folder_counts):
    print("{}: {} images".format(folder, folder_counts[folder]))

Files in the repository: 10422
Image files: 10419
Other files: ['.gitattributes', '.gitignore', 'README.md']
test/CORAL: 974 images
test/CORAL_BL: 591 images
train/CORAL: 4541 images
train/CORAL_BL: 2751 images
val/CORAL: 973 images
val/CORAL_BL: 589 images


### Images in the Drive folder

In [6]:
import time

start = time.time()
rows = []
skipped = []
for root, dirs, names in os.walk(noaa_dir):
    if ".git" in dirs:
        dirs.remove(".git")
    for name in names:
        relative = os.path.relpath(os.path.join(root, name), noaa_dir).replace(os.sep, "/")
        parts = relative.split("/")
        if name.lower().endswith(image_endings) and len(parts) == 3:
            rows.append({"split": parts[0], "label": parts[1], "path": relative})
        else:
            skipped.append(relative)

noaa = pd.DataFrame(rows)
print("Listing took {:.1f} seconds".format(time.time() - start))
print("Images found: {}".format(len(noaa)))
print("Files skipped: {}, first ones: {}".format(len(skipped), skipped[:20]))
print("Splits: {}".format(sorted(noaa["split"].unique())))
print("Labels: {}".format(sorted(noaa["label"].unique())))
print(noaa.head(5))

Listing took 26.9 seconds
Images found: 10419
Files skipped: 3, first ones: ['.gitattributes', '.gitignore', 'README.md']
Splits: ['test', 'train', 'val']
Labels: ['CORAL', 'CORAL_BL']
  split     label                                      path
0   val  CORAL_BL  val/CORAL_BL/LAN-B1820_2019_13_10127.PNG
1   val  CORAL_BL  val/CORAL_BL/LAN-B1824_2019_05_10411.PNG
2   val  CORAL_BL   val/CORAL_BL/LAN-B1800_2019_19_9263.PNG
3   val  CORAL_BL   val/CORAL_BL/LAN-B1808_2019_08_9776.PNG
4   val  CORAL_BL    val/CORAL_BL/KUR-B016_2019_10_6567.PNG


### Row counts against the folders

In [7]:
drive_counts = Counter(noaa["split"] + "/" + noaa["label"])

for folder in sorted(set(folder_counts) | set(drive_counts)):
    print("{}: {} on the Hub, {} on Drive, same: {}".format(
        folder, folder_counts[folder], drive_counts[folder], folder_counts[folder] == drive_counts[folder]))
print("Total: {} on the Hub, {} on Drive".format(sum(folder_counts.values()), len(noaa)))

print(pd.crosstab(noaa["split"], noaa["label"]))

test/CORAL: 974 on the Hub, 974 on Drive, same: True
test/CORAL_BL: 591 on the Hub, 591 on Drive, same: True
train/CORAL: 4541 on the Hub, 4541 on Drive, same: True
train/CORAL_BL: 2751 on the Hub, 2751 on Drive, same: True
val/CORAL: 973 on the Hub, 973 on Drive, same: True
val/CORAL_BL: 589 on the Hub, 589 on Drive, same: True
Total: 10419 on the Hub, 10419 on Drive
label  CORAL  CORAL_BL
split                 
test     974       591
train   4541      2751
val      973       589


### Paths that differ between the Hub and Drive

In [8]:
import re

hub_paths = set(image_files)
drive_paths = set(noaa["path"])

only_on_drive = sorted(drive_paths - hub_paths)
only_on_hub = sorted(hub_paths - drive_paths)

print("On Drive but not on the Hub: {}".format(len(only_on_drive)))
for path in only_on_drive[:50]:
    print("  {}".format(path))

print("On the Hub but not on Drive: {}".format(len(only_on_hub)))
for path in only_on_hub[:50]:
    print("  {}".format(path))

# A copy suffix such as " (1)" before the file ending is removed first
base_names = {}
for path in sorted(drive_paths):
    name = path.split("/")[-1]
    base = re.sub(r" \(\d+\)(\.\w+)$", r"\1", name)
    base_names.setdefault(base, []).append(path)

repeated = {base: paths for base, paths in base_names.items() if len(paths) > 1}
print("File names that appear more than once on Drive: {}".format(len(repeated)))
for base in sorted(repeated)[:50]:
    print("  {}: {}".format(base, repeated[base]))

duplicate_paths = noaa[noaa["path"].duplicated(keep=False)]
print("Rows with the same path listed more than once: {}".format(len(duplicate_paths)))
print(duplicate_paths.head(10))

On Drive but not on the Hub: 0
On the Hub but not on Drive: 0
File names that appear more than once on Drive: 0
Rows with the same path listed more than once: 0
Empty DataFrame
Columns: [split, label, path]
Index: []


**Observation**: All 10,419 images in Google Drive match the Hugging Face repository counts across all six split and class folders. No missing files, extra files, or duplicate paths were found, confirming that the dataset copy is consistent and images can be labelled using their folder names (CORAL and CORAL_BL).